# Already-solved pairs in the official LeWM Cube dataset

1. Upload `grounded-jepa-system1.zip` to `MyDrive/hybrid-wm/` (as for the GPU study), replacing the old one.
2. Runtime: a GPU runtime (its rendering setup is known to work) with at least 100 GB of free disk: the
   download is about 46 GB and is streamed, but the `.h5` file it holds is 95 GB and must be written to disk.
   The first cell prints the free disk; stop there if it shows less than 100 GB.
3. The result is saved to `MyDrive/hybrid-wm/official_cube_check.json` as soon as the already-solved share is
   computed, and again after the authors' random-policy evaluation, which runs 50 environments in parallel and
   needs about 9 GB of RAM. If that step runs out of memory, the file records the error and still holds the share.
   Send the file back either way.


In [ ]:
import os, subprocess, sys, shutil, pathlib
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/hybrid-wm")
run = lambda cmd, **kw: subprocess.run(cmd, check=True, **kw)
run(["unzip", "-q", "-o", str(DRIVE / "grounded-jepa-system1.zip"), "-d", "/content"])
SUITE = pathlib.Path("/content/grounded-jepa-system1/bench/lewm_suite")
pip = [sys.executable, "-m", "pip", "install", "-q"]
run(pip + ["git+https://github.com/galilai-group/stable-worldmodel@21446f1ede6d5284e981bd7b47f432b994e6d812"])
run(pip + ["stable-pretraining==0.1.8", "mujoco==3.15.0", "h5py", "hdf5plugin", "zstandard", "huggingface_hub",
           "hydra-core", "scikit-learn", "imageio[ffmpeg]", "pygame", "pymunk", "shapely", "dm-env", "dm-tree",
           "absl-py", "lxml", "PyOpenGL", "glfw"])
run(pip + ["--no-deps", "ogbench==1.2.1", "dm_control==1.0.48"])
if not pathlib.Path("/content/le-wm").exists():
    run(["git", "clone", "https://github.com/lucas-maes/le-wm", "/content/le-wm"])
run(["git", "-C", "/content/le-wm", "checkout", "-q", "8edfeb336732b5f3ce7b8b210d0ba370a09e2cac"])
print("free disk GB:", shutil.disk_usage("/content").free // 2**30,
      "| RAM GB:", os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") // 2**30)

In [ ]:
os.environ["MUJOCO_GL"] = "egl"
out = DRIVE / "official_cube_check.json"
run([sys.executable, str(SUITE / "official_cube_check.py"), "--out", str(out), "--random-eval",
     "--lewm-repo", "/content/le-wm", "--data-dir", "/content/lewm_cube", "--home", "/content/swm_home_official"])
print(out.read_text()[:1500])